# Modul 13: Klassische Bild- und Signalmodelle

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Klassische Bildmodelle, Signalmodelle  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschritten  
    **Orientierungszeit:** etwa 135 bis 185 Minuten

    ## Überblick

    Sie erzeugen klassische Merkmale aus Bildern und Signalfenstern und trainieren scikit-learn-Pipelines darauf. Leakage-freie Splits, zeitliche Bewertung, Konfusionsmatrix, Fehlbilder und Vorhersageverläufe verbinden beide Datentypen.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_13A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_13B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Kleine Bilddaten als Pixelvektoren oder klassische Bildmerkmale vorbereiten.
- Klassische Klassifikatoren auf Bildmerkmalen ohne Datenleckage trainieren.
- Bildmodelle mit Konfusionsmatrix und Fehlklassifikationen bewerten.
- Fenster und Statistik- sowie Frequenzmerkmale aus Signalen erzeugen.
- scikit-learn-Pipelines für Signal-Klassifikation oder Regression trainieren.
- Signalmodelle zeitlich korrekt bewerten und mit Baselines vergleichen.

    ## Bewertete Fähigkeiten

    - Digits, Pixelvektoren, Histogramme, Kanten und HOG
- PCA in Pipelines, LogisticRegression und SVC
- Fensterlabels, Statistik- und FFT-Merkmale
- zeitlicher Split, TimeSeriesSplit und Vorhersagevisualisierung

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from skimage.feature import hog

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

digits_13 = load_digits()
images_13 = digits_13.images.astype(np.float32)
labels_13 = digits_13.target

# Kontinuierliches Signal mit wechselnden Zuständen für eine zeitliche Aufgabe.
sampling_rate_13 = 50
segment_length_13 = 100
state_frequencies_13 = [3, 3, 8, 8, 3, 8, 3, 8, 8, 3, 3, 8]
signal_parts_13 = []
state_labels_13 = []
for state_index, frequency in enumerate(state_frequencies_13):
    local_time = np.arange(segment_length_13) / sampling_rate_13
    part = (
        np.sin(2 * np.pi * frequency * local_time)
        + 0.25 * np.sin(2 * np.pi * (frequency + 2) * local_time)
        + rng.normal(0, 0.18, segment_length_13)
    )
    signal_parts_13.append(part)
    state_labels_13.extend([0 if frequency == 3 else 1] * segment_length_13)
continuous_signal_13 = np.concatenate(signal_parts_13)
point_labels_13 = np.array(state_labels_13)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Pixel-, Histogramm-, Kanten- und HOG-Merkmale erzeugen

    Erstellen Sie für jedes Digits-Bild vier Merkmalsdarstellungen:

1. rohe 64 Pixelwerte,
2. Intensitätshistogramm mit acht Bins,
3. einfache Kantenmerkmale aus horizontalen und vertikalen Differenzen,
4. HOG-Merkmale mit für 8x8-Bilder geeigneten kleinen Zellen.

Geben Sie Formen und erste Merkmalszeilen aus. Visualisieren Sie ein Originalbild und seine Gradientenstärke in getrennten Abbildungen.

> **Hinweis:** Prüfen Sie, ob jedes Bild genau eine Merkmalszeile erzeugt.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Prüfen Sie, ob jedes Bild genau eine Merkmalszeile erzeugt.

## Aufgabe 2: Bildklassifikatoren leakage-frei vergleichen

    Erzeugen Sie einmalig stratifizierte Train/Test-Indizes und wenden Sie dieselben Indizes auf alle Merkmalsmatrizen an.

Vergleichen Sie:

- Dummy-Baseline,
- skalierte logistische Regression auf Rohpixeln,
- skalierte logistische Regression auf HOG,
- PCA(20) + skalierte logistische Regression auf Rohpixeln,
- skalierte SVC auf HOG.

Berechnen Sie Testgenauigkeit und speichern Sie Vorhersagen.

> **Hinweis:** Feature-Extraktion ohne gelernte Parameter darf vor dem Split erfolgen; gelernte PCA dagegen nicht.

In [ ]:
all_indices = np.arange(len(labels_13))

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Feature-Extraktion ohne gelernte Parameter darf vor dem Split erfolgen; gelernte PCA dagegen nicht.

## Aufgabe 3: Konfusionsmatrix, PCA und Fehlklassifikationen analysieren

    Wählen Sie das beste Bildmodell aus Aufgabe 2.

1. Erstellen Sie seine Konfusionsmatrix als DataFrame.
2. Identifizieren Sie die fünf häufigsten Verwechslungspaare außerhalb der Diagonale.
3. Zeigen Sie bis zu zwölf falsch klassifizierte Bilder in einer gemeinsamen Matplotlib-Abbildung mit wahrem und vorhergesagtem Label.
4. Falls das PCA-Modell trainiert wurde, geben Sie kumulierte erklärte Varianz der 20 Komponenten aus.

> **Hinweis:** Analysieren Sie sowohl häufige Verwechslungspaare als auch konkrete Bilder.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Analysieren Sie sowohl häufige Verwechslungspaare als auch konkrete Bilder.

## Aufgabe 4: Signalfenster und zeitlich geordnete Merkmalstabelle erzeugen

    Zerlegen Sie `continuous_signal_13` in nicht überlappende Fenster der Länge 100. Für jedes Fenster:

- übernehmen Sie das Mehrheitslabel der Punktlabels,
- berechnen Sie Mittelwert, Standardabweichung, RMS, Peak-to-Peak, maximale Änderung,
- berechnen Sie dominante Frequenz und Spektralenergie in 0 bis 5 Hz sowie 5 bis 15 Hz,
- speichern Sie Startzeit und Endzeit.

Erstellen Sie eine nach Zeit sortierte Merkmalstabelle und prüfen Sie die Klassenverteilung.

> **Hinweis:** Die Zielspalte gehört nicht in die Merkmalsmatrix.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Die Zielspalte gehört nicht in die Merkmalsmatrix.

## Aufgabe 5: Integrationsaufgabe: Signalpipeline mit zeitlicher Bewertung

    Verwenden Sie die Merkmalstabelle aus Aufgabe 4.

1. Nehmen Sie die ersten acht Fenster als Training und die letzten vier als Test.
2. Vergleichen Sie Mehrheitsklassen-Dummy, skalierte logistische Regression und skalierte SVC.
3. Führen Sie zusätzlich `TimeSeriesSplit(n_splits=4)` nur auf dem Trainingsabschnitt durch.
4. Erstellen Sie eine Zeitdarstellung aus wahrem und vorhergesagtem Testzustand.
5. Dokumentieren Sie, warum eine zufällige Kreuzvalidierung hier irreführend sein könnte.

> **Hinweis:** Zeitliche Reihenfolge gehört zur Datenentstehung und damit zur Bewertungslogik.

In [ ]:
# Führen Sie Aufgabe 4 zuerst aus.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Zeitliche Reihenfolge gehört zur Datenentstehung und damit zur Bewertungslogik.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.